# Lead CRM – Table Designing / Data Splitting

This notebook adapts the structure of the supplied sample `MainProject-TableDesigning` notebook to the actual Lead CRM CSV.

In [13]:
import pandas as pd
import numpy as np

CSV_PATH = CSV_PATH = r"C:\Users\user\Desktop\Group project\Data set\CRM_Leads_Cleaned_2.csv"
df = pd.read_csv(CSV_PATH)

print("Raw shape:", df.shape)

Raw shape: (10000, 52)


## 1. Standardize empty/placeholder values

In [14]:
PLACEHOLDERS = ["--Select--", "---Select---", "--None--", "-Other-"]
df = df.replace(PLACEHOLDERS, np.nan)

all_empty_cols = [c for c in df.columns if df[c].isna().all()]
print("Completely empty columns:", all_empty_cols)

df = df.drop(columns=all_empty_cols)
print("Shape after removing empty columns:", df.shape)

Completely empty columns: []
Shape after removing empty columns: (10000, 52)


## 2. Clean date and text fields

In [15]:
date_columns = [
    "Lead Date", "Won Date", "Expected Closing Date",
    "Lead Assigned At", "First Touch At", "Last Call Time",
    "Next Follow-up On", "Tomorrow closing",
    "Created Time", "Updated Time", "Viewed Time", "Droped Date"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()

print("Date and text preparation completed.")

C:\Users\user\AppData\Local\Temp\ipykernel_30424\550894560.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\user\AppData\Local\Temp\ipykernel_30424\550894560.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\user\AppData\Local\Temp\ipykernel_30424\550894560.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
C:\Users\user\AppData\Local\Temp\ipykernel_304

Date and text preparation completed.


## 3. Create `dim_user`

In [16]:
dim_user = (
    df[["Lead Owner"]].dropna().drop_duplicates()
    .sort_values("Lead Owner").reset_index(drop=True)
)
dim_user.insert(0, "user_id", range(1, len(dim_user) + 1))

display(dim_user)
print("Rows:", len(dim_user))

,user_id,Lead Owner
0,1,Afees
1,2,Ambili
2,3,Bridgeon Admin
3,4,Inbound User
4,5,Joel
5,6,Muhammed
6,7,Najiya
7,8,Rincy
8,9,Safa
9,10,Sanju


Rows: 15


## 4. Create `dim_source`

In [17]:
dim_source = (
    df[["Lead Source"]].dropna().drop_duplicates()
    .sort_values("Lead Source").reset_index(drop=True)
)
dim_source.insert(0, "source_id", range(1, len(dim_source) + 1))

display(dim_source)
print("Rows:", len(dim_source))

,source_id,Lead Source
0,1,Camp-Bridge
1,2,Cold Calling
2,3,Facebook Ad
3,4,Google Ad
4,5,Incoming Call
5,6,Incoming Call (Ad)
6,7,Insta Ad
7,8,Insta Ad Chat
8,9,Insta Chat
9,10,Kodlar CRM Tech Lead


Rows: 18


## 5. Create `dim_campaign`

In [18]:
campaign_cols = [
    "Campaign Name", "Campaign Term", "Campaign Content",
    "ad set name", "ad name", "UTM"
]

dim_campaign = (
    df[campaign_cols].dropna(how="all").drop_duplicates()
    .reset_index(drop=True)
)
dim_campaign.insert(0, "campaign_id", range(1, len(dim_campaign) + 1))

display(dim_campaign.head())
print("Rows:", len(dim_campaign))

,campaign_id,Campaign Name,Campaign Term,Campaign Content,ad set name,ad name,UTM
0,1,No Campaign,Not Provided,Not Provided,No Ad Set,No Ad,Not Provided
1,2,STRATEGY-AD01,Not Provided,Not Provided,DT-NK-CC01-STRTG,STR-AD04-DT-NK,Not Provided
2,3,Jul-01,Not Provided,Not Provided,AD01-JULY-LL-CNVTD,CC-03-JUL-LL,Not Provided
3,4,LOCATION-ADS,Not Provided,Not Provided,KOCHI-DT,KOCHI-DT-2,Not Provided
4,5,LEAD AD,Not Provided,Not Provided,Tech-NK-DT - II,Degree Urapichavar (Tech-NK-DT-II),Not Provided


Rows: 158


## 6. Create `dim_contact`

In [19]:
contact_cols = [
    "Contact Reference", "Email Address", "College", "Program",
    "Year of Pass", "District", "Preferred Location",
    "Domain Interested", "Coding Basics"
]

dim_contact = (
    df[contact_cols].drop_duplicates("Contact Reference")
    .reset_index(drop=True)
)
dim_contact.insert(0, "contact_id", range(1, len(dim_contact) + 1))

display(dim_contact.head())
print("Rows:", len(dim_contact))
print("Duplicate Contact Reference:", dim_contact["Contact Reference"].duplicated().sum())

,contact_id,Contact Reference,Email Address,College,Program,Year of Pass,District,Preferred Location,Domain Interested,Coding Basics
0,1,8921366393,not provided,Unknown,Unknown,Unknown,Other,Other,Python Full Stack,No
1,2,7403618555,not provided,Unknown,Unknown,Unknown,Other,Other,Python Full Stack,No
2,3,6282602017,not provided,Unknown,Unknown,Unknown,Other,Other,Python Full Stack,No
3,4,9567163612,not provided,Unknown,Unknown,Unknown,Other,Other,Python Full Stack,No
4,5,8089865717,not provided,Unknown,Unknown,Unknown,Other,Other,Python Full Stack,No


Rows: 10000
Duplicate Contact Reference: 0


## 7. Create `fact_lead`

In [20]:
# Create a surrogate lead_id because the source Lead ID column is completely empty.
fact_lead = df.copy()
fact_lead.insert(0, "lead_id", range(1, len(fact_lead) + 1))

# Foreign keys
fact_lead = fact_lead.merge(
    dim_user, left_on="Lead Owner", right_on="Lead Owner", how="left"
)
fact_lead = fact_lead.merge(
    dim_source, left_on="Lead Source", right_on="Lead Source", how="left"
)
fact_lead = fact_lead.merge(
    dim_contact[["contact_id", "Contact Reference"]],
    on="Contact Reference", how="left"
)

# Campaign matching: use all campaign attributes.
fact_lead = fact_lead.merge(
    dim_campaign,
    on=["Campaign Name", "Campaign Term", "Campaign Content", "ad set name", "ad name", "UTM"],
    how="left"
)

lead_columns = [
    "lead_id", "contact_id", "user_id", "source_id", "campaign_id",
    "Lead Date", "Won Date", "Expected Closing Date", "Lead Stage",
    "Won Revenue", "Qualification", "Won Description", "Lost Reason",
    "Lost Description", "Update few words about this lead", "Lead Title",
    "Lead Assigned At", "First Touch At", "Lead Quality",
    "Tomorrow closing", "Created By", "Created Username", "Created Time",
    "Updated Username", "Updated Time", "Viewed Username", "Viewed Time"
]

fact_lead = fact_lead[lead_columns].copy()

display(fact_lead.head())
print("Rows:", len(fact_lead))
print("Unique lead_id:", fact_lead["lead_id"].nunique())

,lead_id,contact_id,user_id,source_id,campaign_id,Lead Date,Won Date,Expected Closing Date,Lead Stage,Won Revenue,...,First Touch At,Lead Quality,Tomorrow closing,Created By,Created Username,Created Time,Updated Username,Updated Time,Viewed Username,Viewed Time
0,1,1,3,10,1,2026-04-09,2026-04-09,NaT,CLOSED LOST,0.0,...,NaT,Unknown,NaT,Unknown,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04
1,2,2,3,10,1,2026-04-09,2026-04-09,NaT,CLOSED LOST,0.0,...,NaT,Unknown,NaT,Unknown,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04
2,3,3,3,10,1,2026-04-09,2026-04-09,NaT,CLOSED LOST,0.0,...,NaT,Unknown,NaT,Unknown,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04
3,4,4,3,10,1,2026-04-09,2026-04-09,NaT,CLOSED LOST,0.0,...,NaT,Unknown,NaT,Unknown,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04
4,5,5,3,10,1,2026-04-09,2026-04-09,NaT,CLOSED LOST,0.0,...,NaT,Unknown,NaT,Unknown,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04,Bridgeon Admin,2026-09-04


Rows: 10000
Unique lead_id: 10000


## 8. Create `fact_lead_activity`

In [21]:
activity_columns = [
    "Call DID Status", "Last Call Time", "Caller Name", "Call Duration",
    "Call Feedback", "Next Follow-up On", "Next Follow-up Notes",
    "Call Missed by User"
]

fact_lead_activity = df[activity_columns].copy()
fact_lead_activity.insert(0, "lead_id", range(1, len(df) + 1))
fact_lead_activity.insert(0, "activity_id", range(1, len(df) + 1))

display(fact_lead_activity.head())
print("Rows:", len(fact_lead_activity))

,activity_id,lead_id,Call DID Status,Last Call Time,Caller Name,Call Duration,Call Feedback,Next Follow-up On,Next Follow-up Notes,Call Missed by User
0,1,1,Untouched,NaT,Unknown,0.0,No Information,NaT,No Information,No
1,2,2,Untouched,NaT,Unknown,0.0,No Information,NaT,No Information,No
2,3,3,Untouched,NaT,Unknown,0.0,No Information,NaT,No Information,No
3,4,4,Untouched,NaT,Unknown,0.0,No Information,NaT,No Information,No
4,5,5,Untouched,NaT,Unknown,0.0,No Information,NaT,No Information,No


Rows: 10000


## 9. Rename columns to database-friendly names

In [22]:
import re

def snake_case(name):
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_").lower()
    return name

tables = {
    "dim_user": dim_user,
    "dim_source": dim_source,
    "dim_campaign": dim_campaign,
    "dim_contact": dim_contact,
    "fact_lead": fact_lead,
    "fact_lead_activity": fact_lead_activity
}

for table in tables.values():
    table.columns = [snake_case(c) for c in table.columns]

print({name: table.columns.tolist() for name, table in tables.items()})

{'dim_user': ['user_id', 'lead_owner'], 'dim_source': ['source_id', 'lead_source'], 'dim_campaign': ['campaign_id', 'campaign_name', 'campaign_term', 'campaign_content', 'ad_set_name', 'ad_name', 'utm'], 'dim_contact': ['contact_id', 'contact_reference', 'email_address', 'college', 'program', 'year_of_pass', 'district', 'preferred_location', 'domain_interested', 'coding_basics'], 'fact_lead': ['lead_id', 'contact_id', 'user_id', 'source_id', 'campaign_id', 'lead_date', 'won_date', 'expected_closing_date', 'lead_stage', 'won_revenue', 'qualification', 'won_description', 'lost_reason', 'lost_description', 'update_few_words_about_this_lead', 'lead_title', 'lead_assigned_at', 'first_touch_at', 'lead_quality', 'tomorrow_closing', 'created_by', 'created_username', 'created_time', 'updated_username', 'updated_time', 'viewed_username', 'viewed_time'], 'fact_lead_activity': ['activity_id', 'lead_id', 'call_did_status', 'last_call_time', 'caller_name', 'call_duration', 'call_feedback', 'next_fol

## 10. Validate the relational tables

In [23]:
for name, table in tables.items():
    print("\n==============================")
    print(name)
    print("==============================")
    print("Rows:", len(table))
    print("Columns:", len(table.columns))
    print("Duplicate rows:", table.duplicated().sum())
    print("Missing values:", int(table.isna().sum().sum()))

print("\nPrimary-key checks")
for name, col in [
    ("dim_user", "user_id"), ("dim_source", "source_id"),
    ("dim_campaign", "campaign_id"), ("dim_contact", "contact_id"),
    ("fact_lead", "lead_id"), ("fact_lead_activity", "activity_id")
]:
    print(name, tables[name][col].is_unique)


dim_user
Rows: 15
Columns: 2
Duplicate rows: 0
Missing values: 0

dim_source
Rows: 18
Columns: 2
Duplicate rows: 0
Missing values: 0

dim_campaign
Rows: 158
Columns: 7
Duplicate rows: 0
Missing values: 0

dim_contact
Rows: 10000
Columns: 10
Duplicate rows: 0
Missing values: 0

fact_lead
Rows: 10000
Columns: 27
Duplicate rows: 0
Missing values: 53075

fact_lead_activity
Rows: 10000
Columns: 10
Duplicate rows: 0
Missing values: 17087

Primary-key checks
dim_user True
dim_source True
dim_campaign True
dim_contact True
fact_lead True
fact_lead_activity True


## 11. Export the relational tables

In [24]:
import os

OUTPUT_DIR = r"C:\Users\user\Desktop\Group project\database-data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, table in tables.items():
    table.to_csv(os.path.join(OUTPUT_DIR, f"{name}.csv"), index=False)

print(f"Exported {len(tables)} relational tables to {OUTPUT_DIR}")

Exported 6 relational tables to C:\Users\user\Desktop\Group project\database-data


In [25]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
